In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()

OFFICIAL_SRC = ROOT / "external" / "official" / "src"

if str(OFFICIAL_SRC) not in sys.path:
    sys.path.insert(0, str(OFFICIAL_SRC))

print("Project root:", ROOT)
print("Official source:", OFFICIAL_SRC)
print("Official source exists:", OFFICIAL_SRC.exists())

from tracking_cellmot.io import open_dataset

print("tracking_cellmot import: OK")

路径配置完全正确

In [ ]:
from pathlib import Path

ROOT = Path.cwd()

DATA_ROOT = (
    ROOT
    / "data"
    / "raw"
    / "competition"
    / "biohub-cell-tracking-during-development"
)

TRAIN_DIR = DATA_ROOT / "train"
TEST_DIR = DATA_ROOT / "test"

print("ROOT:", ROOT)
print("DATA_ROOT exists:", DATA_ROOT.exists())
print("TRAIN_DIR exists:", TRAIN_DIR.exists())
print("TEST_DIR exists:", TEST_DIR.exists())

In [ ]:
train_zarrs = sorted(TRAIN_DIR.glob("*.zarr"))
train_geffs = sorted(TRAIN_DIR.glob("*.geff"))
test_zarrs = sorted(TEST_DIR.glob("*.zarr"))

print("Train Zarr:", len(train_zarrs))
print("Train GEFF:", len(train_geffs))
print("Test Zarr:", len(test_zarrs))

print("\nFirst 5 train datasets:")
for path in train_zarrs[:5]:
    print(path.stem)

训练数据和标签是一一对应的,总共有 199 组带标签的训练样本。

In [ ]:
from collections import Counter

prefix_counts = Counter(
    path.stem.split("_")[0]
    for path in train_zarrs
)

print(prefix_counts)

6bba_*   128 个; 44b6_*    71 个

In [ ]:
import zarr
from collections import Counter

shape_counts = Counter()

for path in train_zarrs:
    root = zarr.open(path, mode="r")
    
    # OME-Zarr 的实际图像数据在 level 0
    image = root["0"]
    
    shape_counts[image.shape] += 1

print("Unique shapes:")
for shape, count in shape_counts.items():
    print(shape, "->", count, "datasets")

199 个训练样本在张量尺寸上完全一致

1.时间长度统一: 所有样本都是 T=100,后面做 temporal window 时可以统一设计

2.空间尺寸统一: 所有 volume 都是 64×256×256

3.物理尺度仍然是各向异性的: scale = (1.625, 0.40625, 0.40625)

1 voxel in Z≈4×1 voxel in X/Y

In [ ]:
import numpy as np
import pandas as pd
import zarr

records = []

for path in train_zarrs:
    root = zarr.open(path, mode="r")
    image = root["0"]

    # 为了快，只采样 5 个时间点
    sample_times = [0, 24, 49, 74, 99]

    values = []

    for t in sample_times:
        volume = np.asarray(image[t])
        values.append(volume.reshape(-1))

    values = np.concatenate(values)

    records.append({
        "dataset": path.stem,
        "prefix": path.stem.split("_")[0],
        "mean": float(values.mean()),
        "std": float(values.std()),
        "p50": float(np.percentile(values, 50)),
        "p90": float(np.percentile(values, 90)),
        "p99": float(np.percentile(values, 99)),
        "max": int(values.max()),
    })

stats_df = pd.DataFrame(records)

stats_df.head()

In [ ]:
stats_df.groupby("prefix")[
    ["mean", "std", "p50", "p90", "p99", "max"]
].agg(["mean", "std"])

44b6整体比6bba更亮

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

stats_df.boxplot(
    column="mean",
    by="prefix",
    ax=axes[0]
)
axes[0].set_title("Mean intensity")
axes[0].set_xlabel("Prefix")
axes[0].set_ylabel("Intensity")

stats_df.boxplot(
    column="p90",
    by="prefix",
    ax=axes[1]
)
axes[1].set_title("90th percentile")
axes[1].set_xlabel("Prefix")

stats_df.boxplot(
    column="p99",
    by="prefix",
    ax=axes[2]
)
axes[2].set_title("99th percentile")
axes[2].set_xlabel("Prefix")

plt.suptitle("")
plt.tight_layout()
plt.show()

两组的典型亮度差异非常明显，整个intensity distribution存在domain shift，但不是完全可分的两个distribution

In [ ]:
sample_44b6 = stats_df[
    stats_df["prefix"] == "44b6"
]["dataset"].iloc[0]

sample_6bba = stats_df[
    stats_df["prefix"] == "6bba"
]["dataset"].iloc[0]

print("44b6 sample:", sample_44b6)
print("6bba sample:", sample_6bba)

In [ ]:
from tracking_cellmot.io import open_dataset

ds_44 = open_dataset(
    TRAIN_DIR / sample_44b6,
    normalize=False,
    require_tracks=True,
)

ds_6 = open_dataset(
    TRAIN_DIR / sample_6bba,
    normalize=False,
    require_tracks=True,
)

In [ ]:
t = 50

img44 = np.asarray(ds_44.image[t]).max(axis=0)
img6 = np.asarray(ds_6.image[t]).max(axis=0)

In [ ]:
vmin = min(img44.min(), img6.min())
vmax = max(img44.max(), img6.max())

fig, axes = plt.subplots(1, 2, figsize=(12, 6))

axes[0].imshow(img44, cmap="gray", vmin=vmin, vmax=vmax)
axes[0].set_title(f"44b6 | {sample_44b6}")

axes[1].imshow(img6, cmap="gray", vmin=vmin, vmax=vmax)
axes[1].set_title(f"6bba | {sample_6bba}")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
def percentile_normalize(img, low=1, high=99.5):
    lo = np.percentile(img, low)
    hi = np.percentile(img, high)

    normalized = (img - lo) / (hi - lo + 1e-8)

    return np.clip(normalized, 0, 1)


img44_norm = percentile_normalize(img44)
img6_norm = percentile_normalize(img6)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 6))

axes[0].imshow(img44_norm, cmap="gray", vmin=0, vmax=1)
axes[0].set_title(f"44b6 normalized | {sample_44b6}")

axes[1].imshow(img6_norm, cmap="gray", vmin=0, vmax=1)
axes[1].set_title(f"6bba normalized | {sample_6bba}")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import polars as pl

nodes44 = ds_44.tracks.node_attrs(
    attr_keys=["t", "z", "y", "x"]
)

nodes6 = ds_6.tracks.node_attrs(
    attr_keys=["t", "z", "y", "x"]
)

print("44b6:")
print(nodes44.head())
print("total annotated nodes:", nodes44.height)

print("\n6bba:")
print(nodes6.head())
print("total annotated nodes:", nodes6.height)

In [ ]:
t = 50

nodes44_t = nodes44.filter(pl.col("t") == t)
nodes6_t = nodes6.filter(pl.col("t") == t)

print("44b6 annotated nodes at t=50:", nodes44_t.height)
print("6bba annotated nodes at t=50:", nodes6_t.height)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 7))

axes[0].imshow(img44_norm, cmap="gray", vmin=0, vmax=1)
axes[0].scatter(
    nodes44_t["x"],
    nodes44_t["y"],
    s=40,
    facecolors="none",
    edgecolors="red",
    linewidths=1.5,
)
axes[0].set_title(
    f"44b6 | t={t} | GT nodes={nodes44_t.height}"
)

axes[1].imshow(img6_norm, cmap="gray", vmin=0, vmax=1)
axes[1].scatter(
    nodes6_t["x"],
    nodes6_t["y"],
    s=40,
    facecolors="none",
    edgecolors="red",
    linewidths=1.5,
)
axes[1].set_title(
    f"6bba | t={t} | GT nodes={nodes6_t.height}"
)

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()

annotation density和真实的 cell density 有差异

In [ ]:
import pandas as pd
from tqdm import tqdm

annotation_records = []

for zarr_path in tqdm(
    train_zarrs,
    total=len(train_zarrs),
    desc="Reading GEFF annotations",
):
    name = zarr_path.stem

    ds_meta = open_dataset(
        TRAIN_DIR / name,
        normalize=False,
        require_tracks=True,
        load_image=False,
    )

    nodes = ds_meta.tracks.node_attrs(
        attr_keys=["t", "z", "y", "x"]
    )

    nodes_per_t = (
        nodes
        .group_by("t")
        .len()
        .sort("t")
    )

    annotation_records.append(
        {
            "dataset": name,
            "prefix": name.split("_")[0],
            "total_nodes": nodes.height,
            "mean_nodes_per_frame": nodes.height / 100,
            "annotated_frames": nodes["t"].n_unique(),
            "max_nodes_in_frame": nodes_per_t["len"].max(),
        }
    )

annotation_df = pd.DataFrame(annotation_records)

annotation_df.head()

In [ ]:
annotation_df.groupby("prefix")[
    [
        "total_nodes",
        "mean_nodes_per_frame",
        "annotated_frames",
        "max_nodes_in_frame",
    ]
].agg(["mean", "median", "std", "min", "max"])

6bba 的 GT 标注量系统性高于 44b6 

44b6
├── 图像整体更亮
├── 看起来细胞更密集
├── GT 每帧却比较稀疏
└── 很多 dataset 可能只追踪少数 lineage

6bba
├── 图像整体更暗
├── 视觉上细胞更大、更稀疏
├── GT 每帧标注更多
└── tracking graph 更密集

In [ ]:
time_records = []

for zarr_path in train_zarrs:
    name = zarr_path.stem

    ds_meta = open_dataset(
        TRAIN_DIR / name,
        normalize=False,
        require_tracks=True,
        load_image=False,
    )

    nodes = ds_meta.tracks.node_attrs(
        attr_keys=["t"]
    )

    counts = (
        nodes
        .group_by("t")
        .len()
    )

    count_map = dict(
        zip(
            counts["t"].to_list(),
            counts["len"].to_list(),
        )
    )

    for t in range(100):
        time_records.append({
            "dataset": name,
            "prefix": name.split("_")[0],
            "t": t,
            "nodes": count_map.get(t, 0),
        })

time_df = pd.DataFrame(time_records)

time_df.head()

In [ ]:
mean_time = (
    time_df
    .groupby(["prefix", "t"])["nodes"]
    .mean()
    .reset_index()
)

plt.figure(figsize=(12, 5))

for prefix in ["44b6", "6bba"]:
    subset = mean_time[mean_time["prefix"] == prefix]

    plt.plot(
        subset["t"],
        subset["nodes"],
        label=prefix,
    )

plt.xlabel("Timepoint")
plt.ylabel("Mean annotated nodes per dataset")
plt.title("Annotation density across time")
plt.legend()
plt.show()

In [ ]:
coverage_df = (
    time_df.assign(
        has_annotation=time_df["nodes"] > 0
    )
    .groupby(["prefix", "t"])["has_annotation"]
    .mean()
    .reset_index()
)

plt.figure(figsize=(12, 5))

for prefix in ["44b6", "6bba"]:
    subset = coverage_df[
        coverage_df["prefix"] == prefix
    ]

    plt.plot(
        subset["t"],
        subset["has_annotation"],
        label=prefix,
    )

plt.xlabel("Timepoint")
plt.ylabel("Fraction of datasets with annotation")
plt.title("Annotation coverage across time")
plt.ylim(0, 1.05)
plt.legend()
plt.show()

In [ ]:
conditional_df = (
    time_df[time_df["nodes"] > 0]
    .groupby(["prefix", "t"])["nodes"]
    .mean()
    .reset_index()
)

plt.figure(figsize=(12, 5))

for prefix in ["44b6", "6bba"]:
    subset = conditional_df[
        conditional_df["prefix"] == prefix
    ]

    plt.plot(
        subset["t"],
        subset["nodes"],
        label=prefix,
    )

plt.xlabel("Timepoint")
plt.ylabel("Mean nodes among annotated datasets")
plt.title("Conditional annotation density across time")
plt.legend()
plt.show()

44b6 后期下降主要是 annotation coverage loss:annotation sequence 提前结束

6bba:主要是 graph 本身每帧包含的 GT node 数发生变化
	


| 特征                      | 44b6   | 6bba    |
| ----------------------- | ------ | ------- |
| dataset 数               | 71     | 128     |
| 原始图像                    | 更亮、密集  | 更暗、稀疏   |
| 平均 GT nodes/dataset     | 284    | 884     |
| 每帧 GT nodes             | ~3     | ~9      |
| 时间 coverage             | 后期明显下降 | 几乎完整    |
| conditional nodes/frame | 几乎恒定   | 有明显时间结构 |


In [ ]:
edge_records = []

for zarr_path in train_zarrs:
    name = zarr_path.stem

    ds_meta = open_dataset(
        TRAIN_DIR / name,
        normalize=False,
        require_tracks=True,
        load_image=False,
    )

    graph = ds_meta.tracks

    nodes = graph.node_attrs(
        attr_keys=["t"]
    )

    edges = graph.edge_attrs()

    edge_records.append({
        "dataset": name,
        "prefix": name.split("_")[0],
        "nodes": nodes.height,
        "edges": edges.height,
        "edges_per_node": edges.height / nodes.height,
    })

edge_df = pd.DataFrame(edge_records)

edge_df.head()

In [ ]:
edge_df.groupby("prefix")[
    ["nodes", "edges", "edges_per_node"]
].agg(["mean", "median", "std", "min", "max"])

44b6 的 node 虽少，但相对而言 edge 更充分, 已有 GT 往往形成较长、连续 lineage


6bba 的 GT graph 更宽，但可能由更多独立 lineage/component 组成

In [ ]:
component_records = []

for zarr_path in train_zarrs:
    name = zarr_path.stem

    ds_meta = open_dataset(
        TRAIN_DIR / name,
        normalize=False,
        require_tracks=True,
        load_image=False,
    )

    graph = ds_meta.tracks

    nodes = graph.node_attrs(attr_keys=["t"])
    edges = graph.edge_attrs()

    estimated_components = nodes.height - edges.height

    component_records.append({
        "dataset": name,
        "prefix": name.split("_")[0],
        "nodes": nodes.height,
        "edges": edges.height,
        "estimated_components": estimated_components,
        "nodes_per_component": (
            nodes.height / estimated_components
            if estimated_components > 0
            else float("nan")
        ),
    })

component_df = pd.DataFrame(component_records)

In [ ]:
component_df.groupby("prefix")[
    ["estimated_components", "nodes_per_component"]
].agg(["mean", "median", "std", "min", "max"])

In [ ]:
sample_graph = ds_44.tracks
sample_edges = sample_graph.edge_attrs()

print(sample_edges.columns)
print(sample_edges.head())

In [ ]:
import networkx as nx
import pandas as pd
from tqdm.auto import tqdm

component_records = []

for zarr_path in tqdm(train_zarrs):
    name = zarr_path.stem

    ds_meta = open_dataset(
        TRAIN_DIR / name,
        normalize=False,
        require_tracks=True,
        load_image=False,
    )

    graph = ds_meta.tracks

    nodes = graph.node_attrs(
        attr_keys=["node_id", "t"]
    )

    edges = graph.edge_attrs()

    # node_id -> timepoint
    node_time = dict(
        zip(
            nodes["node_id"].to_list(),
            nodes["t"].to_list(),
        )
    )

    # Convert lineage graph to undirected graph only for finding components
    G = nx.Graph()

    G.add_nodes_from(nodes["node_id"].to_list())

    G.add_edges_from(
        zip(
            edges["source_id"].to_list(),
            edges["target_id"].to_list(),
        )
    )

    for component_id, component in enumerate(nx.connected_components(G)):
        times = [node_time[node_id] for node_id in component]

        start_t = min(times)
        end_t = max(times)

        span_frames = end_t - start_t + 1
        observed_timepoints = len(set(times))

        component_records.append({
            "dataset": name,
            "prefix": name.split("_")[0],
            "component_id": component_id,
            "component_nodes": len(component),
            "start_t": start_t,
            "end_t": end_t,
            "span_frames": span_frames,
            "observed_timepoints": observed_timepoints,
            "temporal_coverage": observed_timepoints / span_frames,
        })

component_span_df = pd.DataFrame(component_records)

component_span_df.head()

In [ ]:
component_span_df.groupby("prefix")[
    [
        "component_nodes",
        "span_frames",
        "observed_timepoints",
        "temporal_coverage",
    ]
].agg(
    ["mean", "median", "std", "min", "max"]
)

In [ ]:
edge_time_records = []

for zarr_path in tqdm(train_zarrs):
    name = zarr_path.stem

    ds_meta = open_dataset(
        TRAIN_DIR / name,
        normalize=False,
        require_tracks=True,
        load_image=False,
    )

    graph = ds_meta.tracks

    nodes = graph.node_attrs(
        attr_keys=["node_id", "t"]
    )

    edges = graph.edge_attrs()

    node_time = dict(
        zip(
            nodes["node_id"].to_list(),
            nodes["t"].to_list(),
        )
    )

    for source_id, target_id in zip(
        edges["source_id"].to_list(),
        edges["target_id"].to_list(),
    ):
        source_t = node_time[source_id]
        target_t = node_time[target_id]

        edge_time_records.append({
            "dataset": name,
            "prefix": name.split("_")[0],
            "source_t": source_t,
            "target_t": target_t,
            "delta_t": target_t - source_t,
        })

edge_time_df = pd.DataFrame(edge_time_records)

In [ ]:
edge_time_df["delta_t"].value_counts().sort_index()

In [ ]:
edge_time_df.groupby("prefix")["delta_t"].value_counts().sort_index()

全部 128,883 条 GT edge 都严格满足 Δt=1



In [ ]:
outdegree_records = []

for zarr_path in tqdm(train_zarrs):
    name = zarr_path.stem

    ds_meta = open_dataset(
        TRAIN_DIR / name,
        normalize=False,
        require_tracks=True,
        load_image=False,
    )

    graph = ds_meta.tracks

    nodes = graph.node_attrs(
        attr_keys=["node_id", "t"]
    )

    edges = graph.edge_attrs()

    # Count outgoing edges for every source node
    out_counts = (
        edges
        .group_by("source_id")
        .len()
    )

    outdegree_map = dict(
        zip(
            out_counts["source_id"].to_list(),
            out_counts["len"].to_list(),
        )
    )

    for node_id, t in zip(
        nodes["node_id"].to_list(),
        nodes["t"].to_list(),
    ):
        outdegree_records.append({
            "dataset": name,
            "prefix": name.split("_")[0],
            "node_id": node_id,
            "t": t,
            "out_degree": outdegree_map.get(node_id, 0),
        })

outdegree_df = pd.DataFrame(outdegree_records)

In [ ]:
outdegree_df["out_degree"].value_counts().sort_index()

division 大约只占：∼0.11%

-> division 是一个极端 class-imbalance 问题

GT graph
│
├── 所有 edge 都严格 t → t+1
│
├── ~96–98% nodes
│      ↓
│   普通 continuation
│
├── ~2–4%
│      ↓
│   track endings
│
└── ~0.11%
       ↓
    division parents


In [ ]:
pd.crosstab(
    outdegree_df["prefix"],
    outdegree_df["out_degree"],
    normalize="index"
)

In [ ]:
division_df = (
    outdegree_df[
        outdegree_df["out_degree"] == 2
    ]
    .copy()
)

print("Total divisions:", len(division_df))

print(
    division_df.groupby("prefix")
    .size()
)

In [ ]:
plt.figure(figsize=(12, 5))

for prefix in ["44b6", "6bba"]:
    subset = division_df[
        division_df["prefix"] == prefix
    ]

    counts = (
        subset["t"]
        .value_counts()
        .sort_index()
    )

    plt.plot(
        counts.index,
        counts.values,
        marker="o",
        label=prefix,
    )

plt.xlabel("Division parent timepoint")
plt.ylabel("Number of GT divisions")
plt.title("GT division events across time")
plt.legend()
plt.show()

混合了两个因素：division count=parent 数量 × division probability

真正应该研究的是：P(division | 一个有下一帧连接的 GT parent): 

在这个时间点所有继续存在的 parent node 中，有多少比例发生了分裂？

In [ ]:
division_rate_time = (
    outdegree_df[
        outdegree_df["out_degree"] >= 1
    ]
    .assign(
        is_division=lambda df: df["out_degree"] == 2
    )
    .groupby(["prefix", "t"])["is_division"]
    .agg(["sum", "count"])
    .reset_index()
)

division_rate_time["division_rate"] = (
    division_rate_time["sum"]
    / division_rate_time["count"]
)

division_rate_time.head()

In [ ]:
division_bin_df = outdegree_df[
    outdegree_df["out_degree"] >= 1
].copy()

division_bin_df["time_bin"] = (
    division_bin_df["t"] // 10
) * 10

division_bin_df["is_division"] = (
    division_bin_df["out_degree"] == 2
)

division_bin_summary = (
    division_bin_df
    .groupby(["prefix", "time_bin"])["is_division"]
    .agg(["sum", "count"])
    .reset_index()
)

division_bin_summary["division_rate"] = (
    division_bin_summary["sum"]
    / division_bin_summary["count"]
)

division_bin_summary

In [ ]:
plt.figure(figsize=(11, 5))

for prefix in ["44b6", "6bba"]:
    subset = division_bin_summary[
        division_bin_summary["prefix"] == prefix
    ]

    plt.plot(
        subset["time_bin"],
        subset["division_rate"],
        marker="o",
        label=prefix,
    )

plt.xlabel("Time bin start")
plt.ylabel("GT division rate")
plt.title("GT division rate across time (10-frame bins)")
plt.legend()
plt.show()

timepoint 本身可能有一点信息，但不是强 division predictor

division model 更应该依赖： cell morphology + motion + daughter geometry
	​


In [ ]:
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

normal_edge_records = []
division_geometry_records = []

for zarr_path in tqdm(train_zarrs):
    name = zarr_path.stem

    ds_meta = open_dataset(
        TRAIN_DIR / name,
        normalize=False,
        require_tracks=True,
        load_image=False,
    )

    graph = ds_meta.tracks
    scale_z, scale_y, scale_x = ds_meta.scale

    nodes = graph.node_attrs(
        attr_keys=["node_id", "t", "z", "y", "x"]
    )

    edges = graph.edge_attrs()

    # node_id -> information
    node_info = {}

    for row in nodes.iter_rows(named=True):
        node_info[row["node_id"]] = {
            "t": row["t"],
            "pos_um": np.array(
                [
                    row["z"] * scale_z,
                    row["y"] * scale_y,
                    row["x"] * scale_x,
                ],
                dtype=float,
            ),
        }

    # source -> targets
    outgoing = {}

    for source_id, target_id in zip(
        edges["source_id"].to_list(),
        edges["target_id"].to_list(),
    ):
        outgoing.setdefault(source_id, []).append(target_id)

    for source_id, target_ids in outgoing.items():

        parent_pos = node_info[source_id]["pos_um"]
        parent_t = node_info[source_id]["t"]

        # Ordinary tracking
        if len(target_ids) == 1:
            child_id = target_ids[0]
            child_pos = node_info[child_id]["pos_um"]

            distance = np.linalg.norm(
                child_pos - parent_pos
            )

            normal_edge_records.append({
                "dataset": name,
                "prefix": name.split("_")[0],
                "t": parent_t,
                "distance_um": distance,
            })

        # Division
        elif len(target_ids) == 2:
            child1_id, child2_id = target_ids

            child1_pos = node_info[child1_id]["pos_um"]
            child2_pos = node_info[child2_id]["pos_um"]

            d_parent_1 = np.linalg.norm(
                child1_pos - parent_pos
            )

            d_parent_2 = np.linalg.norm(
                child2_pos - parent_pos
            )

            d_daughters = np.linalg.norm(
                child1_pos - child2_pos
            )

            division_geometry_records.append({
                "dataset": name,
                "prefix": name.split("_")[0],
                "t": parent_t,

                "parent_daughter_1_um": d_parent_1,
                "parent_daughter_2_um": d_parent_2,

                "mean_parent_daughter_um":
                    (d_parent_1 + d_parent_2) / 2,

                "max_parent_daughter_um":
                    max(d_parent_1, d_parent_2),

                "daughter_daughter_um":
                    d_daughters,
            })

normal_edge_df = pd.DataFrame(normal_edge_records)
division_geometry_df = pd.DataFrame(
    division_geometry_records
)

print("Normal edges:", len(normal_edge_df))
print("Divisions:", len(division_geometry_df))

In [ ]:
normal_edge_df.groupby("prefix")[
    "distance_um"
].describe(
    percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]
)

In [ ]:
division_geometry_df.groupby("prefix")[
    [
        "mean_parent_daughter_um",
        "max_parent_daughter_um",
        "daughter_daughter_um",
    ]
].agg(
    ["mean", "median", "std", "min", "max"]
)

1. 普通 tracking 的位移通常很小
2. Division 后 daughter 离 parent 明显更远
3. 更强的可能是 daughter–daughter separation
4. 普通 edge 有很长的尾部
   
组合：两个合理 daughter candidates+较大的 daughter-daughter separation

Division 有明显的空间几何特征

In [ ]:
plt.figure(figsize=(10, 6))

for prefix in ["44b6", "6bba"]:

    normal = normal_edge_df[
        normal_edge_df["prefix"] == prefix
    ]["distance_um"]

    division = division_geometry_df[
        division_geometry_df["prefix"] == prefix
    ]["mean_parent_daughter_um"]

    plt.hist(
        normal,
        bins=np.linspace(0, 15, 61),
        density=True,
        histtype="step",
        linewidth=2,
        label=f"{prefix} normal",
    )

    plt.hist(
        division,
        bins=np.linspace(0, 15, 61),
        density=True,
        histtype="step",
        linewidth=2,
        label=f"{prefix} division",
    )

plt.xlim(0, 15)
plt.xlabel("Parent → child distance (µm)")
plt.ylabel("Density")
plt.title("Normal tracking vs division geometry")
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(9, 5))

for prefix in ["44b6", "6bba"]:

    subset = division_geometry_df[
        division_geometry_df["prefix"] == prefix
    ]

    plt.hist(
        subset["daughter_daughter_um"],
        bins=np.linspace(0, 22, 45),
        density=True,
        histtype="step",
        linewidth=2,
        label=prefix,
    )

plt.xlabel("Daughter ↔ daughter distance (µm)")
plt.ylabel("Density")
plt.title("Daughter separation at GT divisions")
plt.legend()
plt.show()

两个 daughter 之间通常有一个比较稳定的 separation

In [ ]:
division_symmetry_records = []

for zarr_path in tqdm(train_zarrs):
    name = zarr_path.stem

    ds_meta = open_dataset(
        TRAIN_DIR / name,
        normalize=False,
        require_tracks=True,
        load_image=False,
    )

    graph = ds_meta.tracks
    scale = np.array(ds_meta.scale, dtype=float)

    nodes = graph.node_attrs(
        attr_keys=["node_id", "t", "z", "y", "x"]
    )

    edges = graph.edge_attrs()

    node_pos = {}
    node_time = {}

    for row in nodes.iter_rows(named=True):
        node_id = row["node_id"]

        node_pos[node_id] = np.array(
            [row["z"], row["y"], row["x"]],
            dtype=float,
        ) * scale

        node_time[node_id] = row["t"]

    outgoing = {}

    for source_id, target_id in zip(
        edges["source_id"].to_list(),
        edges["target_id"].to_list(),
    ):
        outgoing.setdefault(source_id, []).append(target_id)

    for parent_id, child_ids in outgoing.items():

        if len(child_ids) != 2:
            continue

        child1_id, child2_id = child_ids

        p = node_pos[parent_id]
        d1 = node_pos[child1_id]
        d2 = node_pos[child2_id]

        v1 = d1 - p
        v2 = d2 - p

        dist1 = np.linalg.norm(v1)
        dist2 = np.linalg.norm(v2)

        # 1. Distance symmetry
        distance_imbalance = (
            abs(dist1 - dist2)
            / ((dist1 + dist2) / 2 + 1e-8)
        )

        # 2. Parent distance from daughter midpoint
        midpoint = (d1 + d2) / 2

        midpoint_offset = np.linalg.norm(
            midpoint - p
        )

        # 3. Angle between daughter directions
        cos_angle = np.dot(v1, v2) / (
            dist1 * dist2 + 1e-8
        )

        cos_angle = np.clip(
            cos_angle,
            -1.0,
            1.0,
        )

        angle_deg = np.degrees(
            np.arccos(cos_angle)
        )

        division_symmetry_records.append({
            "dataset": name,
            "prefix": name.split("_")[0],
            "t": node_time[parent_id],
            "distance_1_um": dist1,
            "distance_2_um": dist2,
            "distance_imbalance": distance_imbalance,
            "midpoint_offset_um": midpoint_offset,
            "daughter_angle_deg": angle_deg,
        })

division_symmetry_df = pd.DataFrame(
    division_symmetry_records
)

division_symmetry_df.head()

1. 两个 daughter 到 parent 的距离并不高度对称
2. 但两个 daughter 确实倾向于位于 parent 的不同方向
3. Parent 也有一定程度位于两个 daughter 之间
   
   具有明显的双向分离几何结构

In [ ]:
division_symmetry_df.groupby("prefix")[
    [
        "distance_imbalance",
        "midpoint_offset_um",
        "daughter_angle_deg",
    ]
].agg(
    ["mean", "median", "std", "min", "max"]
)

In [ ]:
summary_records = []

for prefix in ["44b6", "6bba"]:

    df = division_symmetry_df[
        division_symmetry_df["prefix"] == prefix
    ]

    summary_records.append({
        "prefix": prefix,
        "n_divisions": len(df),

        "angle_gt_90_pct":
            (df["daughter_angle_deg"] > 90).mean(),

        "angle_gt_120_pct":
            (df["daughter_angle_deg"] > 120).mean(),

        "angle_gt_150_pct":
            (df["daughter_angle_deg"] > 150).mean(),

        "imbalance_lt_0.25_pct":
            (df["distance_imbalance"] < 0.25).mean(),

        "imbalance_lt_0.50_pct":
            (df["distance_imbalance"] < 0.50).mean(),

        "midpoint_lt_2um_pct":
            (df["midpoint_offset_um"] < 2).mean(),

        "midpoint_lt_3um_pct":
            (df["midpoint_offset_um"] < 3).mean(),
    })

division_rule_summary = pd.DataFrame(summary_records)

division_rule_summary

大夹角 > parent 接近 daughter 中部 > 两边距离完全相等

In [ ]:
hard_negative_records = []

for zarr_path in tqdm(train_zarrs):
    name = zarr_path.stem

    ds_meta = open_dataset(
        TRAIN_DIR / name,
        normalize=False,
        require_tracks=True,
        load_image=False,
    )

    graph = ds_meta.tracks
    scale = np.array(ds_meta.scale, dtype=float)

    nodes = graph.node_attrs(
        attr_keys=["node_id", "t", "z", "y", "x"]
    )

    edges = graph.edge_attrs()

    node_pos = {}
    node_time = {}
    nodes_by_time = {}

    for row in nodes.iter_rows(named=True):
        node_id = row["node_id"]
        t = row["t"]

        pos = (
            np.array(
                [row["z"], row["y"], row["x"]],
                dtype=float,
            )
            * scale
        )

        node_pos[node_id] = pos
        node_time[node_id] = t

        nodes_by_time.setdefault(t, []).append(node_id)

    outgoing = {}

    for source_id, target_id in zip(
        edges["source_id"].to_list(),
        edges["target_id"].to_list(),
    ):
        outgoing.setdefault(source_id, []).append(target_id)

    for parent_id, child_ids in outgoing.items():

        # Only ordinary tracking parents
        if len(child_ids) != 1:
            continue

        true_child_id = child_ids[0]

        parent_t = node_time[parent_id]
        next_t = parent_t + 1

        if next_t not in nodes_by_time:
            continue

        parent_pos = node_pos[parent_id]
        true_child_pos = node_pos[true_child_id]

        # Other annotated nodes at t+1
        other_ids = [
            node_id
            for node_id in nodes_by_time[next_t]
            if node_id != true_child_id
        ]

        if len(other_ids) == 0:
            continue

        # Find nearest wrong candidate to the parent
        wrong_id = min(
            other_ids,
            key=lambda node_id:
                np.linalg.norm(
                    node_pos[node_id] - parent_pos
                ),
        )

        wrong_pos = node_pos[wrong_id]

        # Restrict to a reasonably local false candidate
        wrong_parent_distance = np.linalg.norm(
            wrong_pos - parent_pos
        )

        if wrong_parent_distance > 15:
            continue

        v1 = true_child_pos - parent_pos
        v2 = wrong_pos - parent_pos

        dist1 = np.linalg.norm(v1)
        dist2 = np.linalg.norm(v2)

        daughter_separation = np.linalg.norm(
            true_child_pos - wrong_pos
        )

        distance_imbalance = (
            abs(dist1 - dist2)
            / ((dist1 + dist2) / 2 + 1e-8)
        )

        midpoint = (
            true_child_pos + wrong_pos
        ) / 2

        midpoint_offset = np.linalg.norm(
            midpoint - parent_pos
        )

        cos_angle = np.dot(v1, v2) / (
            dist1 * dist2 + 1e-8
        )

        cos_angle = np.clip(
            cos_angle,
            -1.0,
            1.0,
        )

        angle_deg = np.degrees(
            np.arccos(cos_angle)
        )

        hard_negative_records.append({
            "dataset": name,
            "prefix": name.split("_")[0],
            "t": parent_t,

            "true_child_distance_um": dist1,
            "wrong_child_distance_um": dist2,

            "daughter_daughter_um":
                daughter_separation,

            "distance_imbalance":
                distance_imbalance,

            "midpoint_offset_um":
                midpoint_offset,

            "daughter_angle_deg":
                angle_deg,
        })

hard_negative_df = pd.DataFrame(
    hard_negative_records
)

print("Hard negative pairs:", len(hard_negative_df))

hard_negative_df.head()

In [ ]:
hard_negative_df.groupby("prefix")[
    [
        "daughter_daughter_um",
        "distance_imbalance",
        "midpoint_offset_um",
        "daughter_angle_deg",
    ]
].agg(
    ["mean", "median", "std", "min", "max"]
)

division 的几何 signature：moderately balanced distances + small midpoint offset + large daughter angle
	​



midpoint / imbalance / angle > daughter separation alone
	​


In [ ]:
from sklearn.metrics import roc_auc_score
import pandas as pd

def geometry_auc(
    positive_df,
    negative_df,
    feature,
    higher_means_division=True,
):
    y_true = (
        [1] * len(positive_df)
        + [0] * len(negative_df)
    )

    scores = pd.concat(
        [
            positive_df[feature],
            negative_df[feature],
        ],
        ignore_index=True,
    ).to_numpy()

    if not higher_means_division:
        scores = -scores

    return roc_auc_score(
        y_true,
        scores,
    )

In [ ]:
for prefix in ["44b6", "6bba"]:

    pos = division_symmetry_df[
        division_symmetry_df["prefix"] == prefix
    ]

    neg = hard_negative_df[
        hard_negative_df["prefix"] == prefix
    ]

    print(f"\n{prefix}")
    print("Positive divisions:", len(pos))
    print("Hard negatives:", len(neg))

    print(
        "Angle AUC:",
        geometry_auc(
            pos,
            neg,
            "daughter_angle_deg",
            higher_means_division=True,
        ),
    )

    print(
        "Midpoint AUC:",
        geometry_auc(
            pos,
            neg,
            "midpoint_offset_um",
            higher_means_division=False,
        ),
    )

    print(
        "Imbalance AUC:",
        geometry_auc(
            pos,
            neg,
            "distance_imbalance",
            higher_means_division=False,
        ),
    )

| Feature                |  44b6 AUC |  6bba AUC | 判断    |
| ---------------------- | --------: | --------: | ----- |
| Angle                  |     0.782 |     0.756 | 有用    |
| **Midpoint offset**    | **0.923** | **0.915** | 非常强   |
| **Distance imbalance** | **0.856** | **0.923** | 强—非常强 |


In [ ]:
features = [
    "distance_imbalance",
    "midpoint_offset_um",
    "daughter_angle_deg",
]

positive_geometry = (
    division_symmetry_df[
        ["dataset", "prefix"] + features
    ]
    .copy()
)

positive_geometry["label"] = 1


negative_geometry = (
    hard_negative_df[
        ["dataset", "prefix"] + features
    ]
    .copy()
)

negative_geometry["label"] = 0


geometry_classification_df = pd.concat(
    [
        positive_geometry,
        negative_geometry,
    ],
    ignore_index=True,
)

geometry_classification_df.groupby(
    ["prefix", "label"]
).size()

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

cv_results = []

for prefix in ["44b6", "6bba"]:

    df = geometry_classification_df[
        geometry_classification_df["prefix"] == prefix
    ].reset_index(drop=True)

    X = df[features]
    y = df["label"]
    groups = df["dataset"]

    cv = StratifiedGroupKFold(
        n_splits=5,
        shuffle=True,
        random_state=42,
    )

    for fold, (train_idx, val_idx) in enumerate(
        cv.split(X, y, groups)
    ):

        model = make_pipeline(
            StandardScaler(),
            LogisticRegression(
                class_weight="balanced",
                max_iter=2000,
            ),
        )

        model.fit(
            X.iloc[train_idx],
            y.iloc[train_idx],
        )

        pred = model.predict_proba(
            X.iloc[val_idx]
        )[:, 1]

        auc = roc_auc_score(
            y.iloc[val_idx],
            pred,
        )

        cv_results.append({
            "prefix": prefix,
            "fold": fold,
            "auc": auc,
            "n_positive_val":
                int(y.iloc[val_idx].sum()),
            "n_negative_val":
                int((y.iloc[val_idx] == 0).sum()),
        })

geometry_cv_df = pd.DataFrame(cv_results)

geometry_cv_df

In [ ]:
geometry_cv_df.groupby("prefix")["auc"].agg(
    ["mean", "std", "min", "max"]
)

ROC-AUC = 0.947 ± 0.019

ROC-AUC 在这个任务里可能过于乐观, 更关心 Precision-Recall
	​


In [ ]:
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
)

oof_records = []

for prefix in ["44b6", "6bba"]:

    df = geometry_classification_df[
        geometry_classification_df["prefix"] == prefix
    ].reset_index(drop=True)

    X = df[features]
    y = df["label"]
    groups = df["dataset"]

    cv = StratifiedGroupKFold(
        n_splits=5,
        shuffle=True,
        random_state=42,
    )

    for fold, (train_idx, val_idx) in enumerate(
        cv.split(X, y, groups)
    ):

        model = make_pipeline(
            StandardScaler(),
            LogisticRegression(
                class_weight="balanced",
                max_iter=2000,
            ),
        )

        model.fit(
            X.iloc[train_idx],
            y.iloc[train_idx],
        )

        pred = model.predict_proba(
            X.iloc[val_idx]
        )[:, 1]

        for idx, score in zip(
            val_idx,
            pred,
        ):
            oof_records.append({
                "prefix": prefix,
                "dataset": df.iloc[idx]["dataset"],
                "label": int(df.iloc[idx]["label"]),
                "score": float(score),
                "fold": fold,
            })


geometry_oof_df = pd.DataFrame(oof_records)

In [ ]:
for prefix in ["44b6", "6bba"]:

    subset = geometry_oof_df[
        geometry_oof_df["prefix"] == prefix
    ]

    y_true = subset["label"]
    y_score = subset["score"]

    prevalence = y_true.mean()

    roc_auc = roc_auc_score(
        y_true,
        y_score,
    )

    ap = average_precision_score(
        y_true,
        y_score,
    )

    print(f"\n{prefix}")
    print(f"Positive prevalence: {prevalence:.4f}")
    print(f"ROC-AUC:             {roc_auc:.4f}")
    print(f"Average Precision:   {ap:.4f}")
    print(f"Random AP baseline:  {prevalence:.4f}")

division geometry 确实具有很强的实际判别信息​

In [ ]:
from sklearn.metrics import precision_recall_curve

pr_operating_records = []

for prefix in ["44b6", "6bba"]:

    subset = geometry_oof_df[
        geometry_oof_df["prefix"] == prefix
    ]

    y_true = subset["label"].to_numpy()
    y_score = subset["score"].to_numpy()

    precision, recall, thresholds = precision_recall_curve(
        y_true,
        y_score,
    )

    pr_df = pd.DataFrame({
        "threshold": thresholds,
        "precision": precision[:-1],
        "recall": recall[:-1],
    })

    print(f"\n{prefix}")

    for target_precision in [0.50, 0.70, 0.80, 0.90, 0.95]:

        valid = pr_df[
            pr_df["precision"] >= target_precision
        ]

        if len(valid) == 0:
            print(
                f"Precision >= {target_precision:.0%}: "
                "not achievable"
            )
            continue

        # Among thresholds satisfying target precision,
        # choose the one with highest recall
        best = valid.loc[
            valid["recall"].idxmax()
        ]

        print(
            f"Precision >= {target_precision:.0%}: "
            f"recall={best['recall']:.3f}, "
            f"threshold={best['threshold']:.3f}"
        )

division 具有明显几何结构，适合作为强辅助 feature，但 geometry-only 不能满足高 precision + 有用 recall

In [ ]:
nn_rank_records = []

for zarr_path in tqdm(train_zarrs):
    name = zarr_path.stem

    ds_meta = open_dataset(
        TRAIN_DIR / name,
        normalize=False,
        require_tracks=True,
        load_image=False,
    )

    graph = ds_meta.tracks
    scale = np.array(ds_meta.scale, dtype=float)

    nodes = graph.node_attrs(
        attr_keys=["node_id", "t", "z", "y", "x"]
    )

    edges = graph.edge_attrs()

    node_pos = {}
    node_time = {}
    nodes_by_time = {}

    for row in nodes.iter_rows(named=True):
        node_id = row["node_id"]
        t = row["t"]

        pos = (
            np.array(
                [row["z"], row["y"], row["x"]],
                dtype=float,
            )
            * scale
        )

        node_pos[node_id] = pos
        node_time[node_id] = t
        nodes_by_time.setdefault(t, []).append(node_id)

    outgoing = {}

    for source_id, target_id in zip(
        edges["source_id"].to_list(),
        edges["target_id"].to_list(),
    ):
        outgoing.setdefault(source_id, []).append(target_id)

    for parent_id, child_ids in outgoing.items():

        # Only ordinary tracking
        if len(child_ids) != 1:
            continue

        true_child_id = child_ids[0]
        next_t = node_time[parent_id] + 1

        candidates = nodes_by_time.get(next_t, [])

        if len(candidates) == 0:
            continue

        parent_pos = node_pos[parent_id]

        candidate_distances = [
            (
                candidate_id,
                np.linalg.norm(
                    node_pos[candidate_id] - parent_pos
                ),
            )
            for candidate_id in candidates
        ]

        candidate_distances.sort(
            key=lambda x: x[1]
        )

        ranked_ids = [
            candidate_id
            for candidate_id, _ in candidate_distances
        ]

        true_rank = (
            ranked_ids.index(true_child_id) + 1
        )

        true_distance = np.linalg.norm(
            node_pos[true_child_id] - parent_pos
        )

        nearest_distance = (
            candidate_distances[0][1]
        )

        nn_rank_records.append({
            "dataset": name,
            "prefix": name.split("_")[0],
            "t": node_time[parent_id],
            "n_candidates": len(candidates),
            "true_rank": true_rank,
            "true_distance_um": true_distance,
            "nearest_distance_um": nearest_distance,
            "nearest_is_true": true_rank == 1,
        })

nn_rank_df = pd.DataFrame(nn_rank_records)

len(nn_rank_df)

In [ ]:
nn_summary = (
    nn_rank_df
    .groupby("prefix")
    .agg(
        n_edges=("true_rank", "size"),
        mean_candidates=("n_candidates", "mean"),
        top1_accuracy=("nearest_is_true", "mean"),
        median_true_rank=("true_rank", "median"),
        top3_accuracy=("true_rank", lambda x: (x <= 3).mean()),
        top5_accuracy=("true_rank", lambda x: (x <= 5).mean()),
    )
)

nn_summary

GT-level association ambiguity 极低；未来 association difficulty 很可能主要来自 detection candidate density，而不是 GT lineage 本身的运动模糊。

In [ ]:
association_margin_records = []

for zarr_path in tqdm(train_zarrs):
    name = zarr_path.stem

    ds_meta = open_dataset(
        TRAIN_DIR / name,
        normalize=False,
        require_tracks=True,
        load_image=False,
    )

    graph = ds_meta.tracks
    scale = np.array(ds_meta.scale, dtype=float)

    nodes = graph.node_attrs(
        attr_keys=["node_id", "t", "z", "y", "x"]
    )

    edges = graph.edge_attrs()

    node_pos = {}
    node_time = {}
    nodes_by_time = {}

    for row in nodes.iter_rows(named=True):
        node_id = row["node_id"]
        t = row["t"]

        node_pos[node_id] = (
            np.array(
                [row["z"], row["y"], row["x"]],
                dtype=float,
            )
            * scale
        )

        node_time[node_id] = t
        nodes_by_time.setdefault(t, []).append(node_id)

    outgoing = {}

    for source_id, target_id in zip(
        edges["source_id"].to_list(),
        edges["target_id"].to_list(),
    ):
        outgoing.setdefault(
            source_id, []
        ).append(target_id)

    for parent_id, child_ids in outgoing.items():

        # ordinary tracking only
        if len(child_ids) != 1:
            continue

        true_child_id = child_ids[0]
        next_t = node_time[parent_id] + 1

        candidates = nodes_by_time.get(next_t, [])

        wrong_candidates = [
            node_id
            for node_id in candidates
            if node_id != true_child_id
        ]

        if len(wrong_candidates) == 0:
            continue

        parent_pos = node_pos[parent_id]

        true_distance = np.linalg.norm(
            node_pos[true_child_id] - parent_pos
        )

        nearest_wrong_distance = min(
            np.linalg.norm(
                node_pos[node_id] - parent_pos
            )
            for node_id in wrong_candidates
        )

        margin = (
            nearest_wrong_distance
            - true_distance
        )

        ratio = (
            nearest_wrong_distance
            / (true_distance + 1e-8)
        )

        association_margin_records.append({
            "dataset": name,
            "prefix": name.split("_")[0],
            "t": node_time[parent_id],
            "true_distance_um": true_distance,
            "nearest_wrong_distance_um":
                nearest_wrong_distance,
            "margin_um": margin,
            "wrong_true_ratio": ratio,
            "nn_failure": margin < 0,
        })

association_margin_df = pd.DataFrame(
    association_margin_records
)

In [ ]:
association_margin_df.groupby("prefix")[
    [
        "true_distance_um",
        "nearest_wrong_distance_um",
        "margin_um",
        "wrong_true_ratio",
    ]
].describe(
    percentiles=[
        0.01,
        0.05,
        0.10,
        0.50,
        0.90,
        0.95,
        0.99,
    ]
)

wrong_true_ratio 被 true_distance_um = 0 的情况炸掉了，所以出现了 1e9、1e10 这种没有实际解释价值的数

In [ ]:
for prefix in ["44b6", "6bba"]:

    df = association_margin_df[
        association_margin_df["prefix"] == prefix
    ]

    print(f"\n{prefix}")

    print("\nTrue child distance:")
    print(
        df["true_distance_um"].quantile(
            [0, 0.01, 0.05, 0.10, 0.50, 0.90, 0.95, 0.99, 1]
        )
    )

    print("\nNearest wrong distance:")
    print(
        df["nearest_wrong_distance_um"].quantile(
            [0, 0.01, 0.05, 0.10, 0.50, 0.90, 0.95, 0.99, 1]
        )
    )

    print("\nMargin:")
    print(
        df["margin_um"].quantile(
            [0, 0.01, 0.05, 0.10, 0.50, 0.90, 0.95, 0.99, 1]
        )
    )

In [ ]:
association_margin_df.groupby("prefix")[
    "nn_failure"
].agg(["sum", "mean"])

原始 3D image

↓

检测出很多 candidate cells

↓

其中大量是真实但没有 GT annotation 的细胞

↓

candidate density 远高于我们现在看到的 5 / 11 个

In [ ]:
nn_failures = (
    association_margin_df[
        association_margin_df["nn_failure"]
    ]
    .copy()
    .sort_values("margin_um")
)

print("Total NN failures:", len(nn_failures))

nn_failures[
    [
        "dataset",
        "prefix",
        "t",
        "true_distance_um",
        "nearest_wrong_distance_um",
        "margin_um",
    ]
].head(20)

In [ ]:
failure_by_dataset = (
    nn_failures
    .groupby(["prefix", "dataset"])
    .size()
    .reset_index(name="n_failures")
    .sort_values(
        "n_failures",
        ascending=False,
    )
)

failure_by_dataset.head(20)

In [ ]:
nn_failures.groupby("prefix")[
    [
        "true_distance_um",
        "nearest_wrong_distance_um",
        "margin_um",
    ]
].describe()

某些 dataset 存在特殊的 motion / drift / annotation 情况
	​


In [ ]:
target_name = "6bba_f20478e9"
target_t = 65

ds_target = open_dataset(
    TRAIN_DIR / target_name,
    normalize=False,
    require_tracks=True,
    load_image=False,
)

graph = ds_target.tracks
scale = np.array(ds_target.scale, dtype=float)

nodes = graph.node_attrs(
    attr_keys=["node_id", "t", "z", "y", "x"]
)

edges = graph.edge_attrs()

node_info = {}

for row in nodes.iter_rows(named=True):
    node_info[row["node_id"]] = {
        "t": row["t"],
        "pos": (
            np.array(
                [row["z"], row["y"], row["x"]],
                dtype=float,
            )
            * scale
        ),
    }

motion_records = []

for source_id, target_id in zip(
    edges["source_id"].to_list(),
    edges["target_id"].to_list(),
):

    if node_info[source_id]["t"] != target_t:
        continue

    p0 = node_info[source_id]["pos"]
    p1 = node_info[target_id]["pos"]

    delta = p1 - p0

    motion_records.append({
        "source_id": source_id,
        "target_id": target_id,
        "dz_um": delta[0],
        "dy_um": delta[1],
        "dx_um": delta[2],
        "distance_um": np.linalg.norm(delta),
    })

motion_df = pd.DataFrame(motion_records)

motion_df

In [ ]:
print("Number of edges:", len(motion_df))

print("\nMedian displacement vector:")
print(
    motion_df[
        ["dz_um", "dy_um", "dx_um"]
    ].median()
)

print("\nDistance summary:")
print(
    motion_df["distance_um"].describe()
)

In [ ]:
motion_df[
    ["dz_um", "dy_um", "dx_um", "distance_um"]
].sort_values(
    "distance_um",
    ascending=False,
)

猜测：embryo/sample/FOV 的整体轴向运动或 acquisition shift

In [ ]:
global_motion_records = []

for zarr_path in tqdm(train_zarrs):
    name = zarr_path.stem

    ds_meta = open_dataset(
        TRAIN_DIR / name,
        normalize=False,
        require_tracks=True,
        load_image=False,
    )

    graph = ds_meta.tracks
    scale = np.array(ds_meta.scale, dtype=float)

    nodes = graph.node_attrs(
        attr_keys=["node_id", "t", "z", "y", "x"]
    )

    edges = graph.edge_attrs()

    node_pos = {}
    node_time = {}

    for row in nodes.iter_rows(named=True):
        node_id = row["node_id"]

        node_pos[node_id] = (
            np.array(
                [row["z"], row["y"], row["x"]],
                dtype=float,
            )
            * scale
        )

        node_time[node_id] = row["t"]

    outgoing = {}

    for source_id, target_id in zip(
        edges["source_id"].to_list(),
        edges["target_id"].to_list(),
    ):
        outgoing.setdefault(
            source_id, []
        ).append(target_id)

    vectors_by_t = {}

    for source_id, target_ids in outgoing.items():

        # ordinary tracking only
        if len(target_ids) != 1:
            continue

        target_id = target_ids[0]

        t = node_time[source_id]

        delta = (
            node_pos[target_id]
            - node_pos[source_id]
        )

        vectors_by_t.setdefault(
            t, []
        ).append(delta)

    for t, vectors in vectors_by_t.items():

        vectors = np.array(vectors)

        median_vector = np.median(
            vectors,
            axis=0,
        )

        median_motion = np.linalg.norm(
            median_vector
        )

        individual_distances = np.linalg.norm(
            vectors,
            axis=1,
        )

        global_motion_records.append({
            "dataset": name,
            "prefix": name.split("_")[0],
            "t": t,
            "n_edges": len(vectors),

            "median_dz_um":
                median_vector[0],

            "median_dy_um":
                median_vector[1],

            "median_dx_um":
                median_vector[2],

            "median_global_motion_um":
                median_motion,

            "median_individual_motion_um":
                np.median(individual_distances),
        })

global_motion_df = pd.DataFrame(
    global_motion_records
)

In [ ]:
global_motion_df[
    global_motion_df["n_edges"] >= 3
].sort_values(
    "median_global_motion_um",
    ascending=False,
).head(30)

global motion ≈ individual motion

这说明这些 frame 的多个 GT cells 确实在朝相似方向一起移动

In [ ]:
gm = global_motion_df[
    global_motion_df["n_edges"] >= 3
].copy()

gm["coherence_ratio"] = (
    gm["median_global_motion_um"]
    / (
        gm["median_individual_motion_um"]
        + 1e-8
    )
)

gm["coherence_ratio"] = gm[
    "coherence_ratio"
].clip(upper=1.0)

gm.groupby("prefix")[
    [
        "median_global_motion_um",
        "coherence_ratio",
    ]
].describe(
    percentiles=[
        0.50,
        0.90,
        0.95,
        0.99,
    ]
)

In [ ]:
threshold_records = []

for prefix in ["44b6", "6bba"]:

    df = gm[
        gm["prefix"] == prefix
    ]

    for threshold in [
        5,
        10,
        15,
        20,
        30,
    ]:

        mask = (
            (df["median_global_motion_um"] >= threshold)
            &
            (df["coherence_ratio"] >= 0.8)
        )

        threshold_records.append({
            "prefix": prefix,
            "threshold_um": threshold,
            "n_frame_transitions": len(df),
            "n_coherent_jumps": int(mask.sum()),
            "fraction": mask.mean(),
            "n_datasets_affected":
                df.loc[
                    mask,
                    "dataset"
                ].nunique(),
        })

motion_jump_summary = pd.DataFrame(
    threshold_records
)

motion_jump_summary

coherence = ||median displacement vector|| / median individual displacement

local tracking为主 + frame-level motion compensation / fallback (尤其 6bba 更需要)

In [ ]:
motion_residual_records = []

for zarr_path in tqdm(train_zarrs):
    name = zarr_path.stem

    ds_meta = open_dataset(
        TRAIN_DIR / name,
        normalize=False,
        require_tracks=True,
        load_image=False,
    )

    graph = ds_meta.tracks
    scale = np.array(ds_meta.scale, dtype=float)

    nodes = graph.node_attrs(
        attr_keys=["node_id", "t", "z", "y", "x"]
    )

    edges = graph.edge_attrs()

    node_pos = {}
    node_time = {}

    for row in nodes.iter_rows(named=True):
        node_id = row["node_id"]

        node_pos[node_id] = (
            np.array(
                [row["z"], row["y"], row["x"]],
                dtype=float,
            )
            * scale
        )

        node_time[node_id] = row["t"]

    outgoing = {}

    for source_id, target_id in zip(
        edges["source_id"].to_list(),
        edges["target_id"].to_list(),
    ):
        outgoing.setdefault(
            source_id, []
        ).append(target_id)

    # Ordinary tracking edges grouped by source time
    edges_by_t = {}

    for source_id, target_ids in outgoing.items():

        if len(target_ids) != 1:
            continue

        target_id = target_ids[0]
        t = node_time[source_id]

        delta = (
            node_pos[target_id]
            - node_pos[source_id]
        )

        edges_by_t.setdefault(
            t, []
        ).append(
            (source_id, target_id, delta)
        )

    for t, frame_edges in edges_by_t.items():

        # Need enough other edges to estimate frame motion
        if len(frame_edges) < 3:
            continue

        for i, (source_id, target_id, delta) in enumerate(
            frame_edges
        ):

            other_vectors = np.array([
                x[2]
                for j, x in enumerate(frame_edges)
                if j != i
            ])

            global_shift = np.median(
                other_vectors,
                axis=0,
            )

            raw_distance = np.linalg.norm(
                delta
            )

            residual_vector = (
                delta - global_shift
            )

            residual_distance = np.linalg.norm(
                residual_vector
            )

            motion_residual_records.append({
                "dataset": name,
                "prefix": name.split("_")[0],
                "t": t,
                "raw_distance_um": raw_distance,
                "global_shift_um":
                    np.linalg.norm(global_shift),
                "residual_distance_um":
                    residual_distance,
            })

motion_residual_df = pd.DataFrame(
    motion_residual_records
)

In [ ]:
for prefix in ["44b6", "6bba"]:

    df = motion_residual_df[
        motion_residual_df["prefix"] == prefix
    ]

    print(f"\n{prefix}")

    print("RAW")
    print(
        df["raw_distance_um"].quantile(
            [0.5, 0.9, 0.95, 0.99, 1.0]
        )
    )

    print("\nAFTER GLOBAL MOTION CORRECTION")
    print(
        df["residual_distance_um"].quantile(
            [0.5, 0.9, 0.95, 0.99, 1.0]
        )
    )

frame-level global motion correction 确实能明显压缩普通 tracking 的长尾位移

6bba 特别值得加入 global-motion-aware association​

raw coordinates

      ↓

estimate global frame motion

      ↓

motion-compensated expected position

      ↓

local candidate search

      ↓
      
association score

In [ ]:
motion_corrected_nn_records = []

for zarr_path in tqdm(train_zarrs):
    name = zarr_path.stem

    ds_meta = open_dataset(
        TRAIN_DIR / name,
        normalize=False,
        require_tracks=True,
        load_image=False,
    )

    graph = ds_meta.tracks
    scale = np.array(ds_meta.scale, dtype=float)

    nodes = graph.node_attrs(
        attr_keys=["node_id", "t", "z", "y", "x"]
    )

    edges = graph.edge_attrs()

    node_pos = {}
    node_time = {}
    nodes_by_time = {}

    for row in nodes.iter_rows(named=True):
        node_id = row["node_id"]
        t = row["t"]

        pos = (
            np.array(
                [row["z"], row["y"], row["x"]],
                dtype=float,
            )
            * scale
        )

        node_pos[node_id] = pos
        node_time[node_id] = t
        nodes_by_time.setdefault(t, []).append(node_id)

    outgoing = {}

    for source_id, target_id in zip(
        edges["source_id"].to_list(),
        edges["target_id"].to_list(),
    ):
        outgoing.setdefault(
            source_id, []
        ).append(target_id)

    # ordinary tracking edges grouped by time
    frame_edges = {}

    for source_id, target_ids in outgoing.items():

        if len(target_ids) != 1:
            continue

        target_id = target_ids[0]
        t = node_time[source_id]

        delta = (
            node_pos[target_id]
            - node_pos[source_id]
        )

        frame_edges.setdefault(t, []).append(
            (source_id, target_id, delta)
        )

    for t, edges_t in frame_edges.items():

        # Need enough edges for leave-one-out motion estimation
        if len(edges_t) < 3:
            continue

        next_candidates = nodes_by_time.get(
            t + 1, []
        )

        if len(next_candidates) == 0:
            continue

        for i, (
            source_id,
            true_target_id,
            true_delta,
        ) in enumerate(edges_t):

            # Estimate motion from OTHER edges only
            other_vectors = np.array([
                edge[2]
                for j, edge in enumerate(edges_t)
                if j != i
            ])

            global_shift = np.median(
                other_vectors,
                axis=0,
            )

            parent_pos = node_pos[source_id]

            # Raw query position
            raw_query = parent_pos

            # Motion-corrected expected position
            corrected_query = (
                parent_pos + global_shift
            )

            raw_distances = []
            corrected_distances = []

            for candidate_id in next_candidates:

                candidate_pos = node_pos[candidate_id]

                raw_distances.append(
                    (
                        candidate_id,
                        np.linalg.norm(
                            candidate_pos - raw_query
                        ),
                    )
                )

                corrected_distances.append(
                    (
                        candidate_id,
                        np.linalg.norm(
                            candidate_pos - corrected_query
                        ),
                    )
                )

            raw_distances.sort(
                key=lambda x: x[1]
            )

            corrected_distances.sort(
                key=lambda x: x[1]
            )

            raw_ids = [
                x[0] for x in raw_distances
            ]

            corrected_ids = [
                x[0] for x in corrected_distances
            ]

            raw_rank = (
                raw_ids.index(true_target_id) + 1
            )

            corrected_rank = (
                corrected_ids.index(true_target_id) + 1
            )

            motion_corrected_nn_records.append({
                "dataset": name,
                "prefix": name.split("_")[0],
                "t": t,

                "global_shift_um":
                    np.linalg.norm(global_shift),

                "raw_rank": raw_rank,
                "corrected_rank": corrected_rank,

                "raw_top1":
                    raw_rank == 1,

                "corrected_top1":
                    corrected_rank == 1,
            })


motion_corrected_nn_df = pd.DataFrame(
    motion_corrected_nn_records
)

In [ ]:
motion_corrected_nn_df.groupby("prefix").agg(
    n_edges=("raw_rank", "size"),

    raw_top1=(
        "raw_top1",
        "mean",
    ),

    corrected_top1=(
        "corrected_top1",
        "mean",
    ),
)

In [ ]:
for prefix in ["44b6", "6bba"]:

    df = motion_corrected_nn_df[
        motion_corrected_nn_df["prefix"] == prefix
    ]

    recovered = (
        (~df["raw_top1"])
        &
        (df["corrected_top1"])
    ).sum()

    worsened = (
        (df["raw_top1"])
        &
        (~df["corrected_top1"])
    ).sum()

    still_wrong = (
        (~df["raw_top1"])
        &
        (~df["corrected_top1"])
    ).sum()

    print(f"\n{prefix}")
    print("Recovered failures:", recovered)
    print("New failures introduced:", worsened)
    print("Still wrong:", still_wrong)

所以对 6bba 来说 rame-level motion 是 association hard cases 的主要来源之一
	​


In [ ]:
motion_corrected_nn_df["motion_bin"] = pd.cut(
    motion_corrected_nn_df["global_shift_um"],
    bins=[
        0,
        1,
        2,
        3,
        5,
        10,
        20,
        np.inf,
    ],
    right=False,
)

motion_bin_summary = (
    motion_corrected_nn_df
    .groupby(
        ["prefix", "motion_bin"],
        observed=True,
    )
    .agg(
        n_edges=("raw_top1", "size"),

        raw_accuracy=(
            "raw_top1",
            "mean",
        ),

        corrected_accuracy=(
            "corrected_top1",
            "mean",
        ),

        raw_failures=(
            "raw_top1",
            lambda x: (~x).sum(),
        ),

        corrected_failures=(
            "corrected_top1",
            lambda x: (~x).sum(),
        ),
    )
    .reset_index()
)

motion_bin_summary["net_failures_removed"] = (
    motion_bin_summary["raw_failures"]
    - motion_bin_summary["corrected_failures"]
)

motion_bin_summary

In [ ]:
transition_records = []

for (prefix, motion_bin), df in (
    motion_corrected_nn_df
    .groupby(
        ["prefix", "motion_bin"],
        observed=True,
    )
):

    recovered = (
        (~df["raw_top1"])
        &
        df["corrected_top1"]
    ).sum()

    worsened = (
        df["raw_top1"]
        &
        (~df["corrected_top1"])
    ).sum()

    transition_records.append({
        "prefix": prefix,
        "motion_bin": motion_bin,
        "n_edges": len(df),
        "recovered": recovered,
        "worsened": worsened,
        "net_gain": recovered - worsened,
    })

motion_transition_df = pd.DataFrame(
    transition_records
)

motion_transition_df


gt ≳ 5 μm : global motion correction 非常重要

association failure 的一个主要机制就是 frame-level motion​, conditional / motion-aware correction 更合理​

普通情况：

cell motion 很小

        ↓

nearest neighbor 几乎完美

异常情况：

embryo / FOV / sample 整体移动

        ↓

raw parent position 不再是好搜索中心

        ↓

nearest neighbor 失败

解决：

estimate global motion ĝ_t

        ↓

expected position = p_t + ĝ_t

        ↓
        
在 expected position 周围做 local matching

In [ ]:
policy_records = []

thresholds = [
    0,      # always corrected
    1,
    2,
    3,
    5,
    10,
    np.inf, # always raw
]

for prefix in ["44b6", "6bba"]:

    df = motion_corrected_nn_df[
        motion_corrected_nn_df["prefix"] == prefix
    ]

    for threshold in thresholds:

        use_corrected = (
            df["global_shift_um"] >= threshold
        )

        policy_correct = np.where(
            use_corrected,
            df["corrected_top1"],
            df["raw_top1"],
        )

        n_failures = int(
            (~policy_correct).sum()
        )

        accuracy = (
            policy_correct.mean()
        )

        policy_records.append({
            "prefix": prefix,
            "threshold_um": threshold,
            "n_edges": len(df),
            "n_failures": n_failures,
            "accuracy": accuracy,
        })

policy_df = pd.DataFrame(
    policy_records
)

policy_df

Association EDA 最终结论


普通 GT association
->
Raw nearest-neighbor
44b6 ≈ 99.92%
6bba ≈ 99.86%
        ->
真正错误主要集中于异常大位移
        ->
这些大位移经常是多个 cell 同方向运动
        ->
存在 frame-level coherent/global motion
        ->
扣除 oracle global motion 后
6bba NN error 减少 ~74%